# Verificar el conjunto de desarrollo transportado por fragmentos

Selecciona CPU, monta Drive e indica la carpeta que contiene `transport.json`, el módulo de transporte, los cuatro fragmentos y los soportes. Se reconstruye y verifica únicamente el paquete de desarrollo en almacenamiento temporal. El paquete reservado de prueba permanece separado.

Ejecuta las celdas en orden. Esta libreta no entrena, no imputa y no aplica normalización específica de modelos.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, importlib.util, json, os, re, sys, tempfile, uuid

if os.environ.get('COLAB_GPU') == '1' or any(
    os.environ.get(key) for key in ('COLAB_TPU_ADDR', 'TPU_NAME', 'COLAB_TPU_TASK_ID')):
    raise RuntimeError('Selecciona CPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución.')
drive.mount('/content/drive')
REMOTE_PACKAGE_DIR = Path(input('Carpeta de Drive que contiene transport.json y los cuatro fragmentos: ').strip()).absolute()
def bootstrap_path(path):
    path = Path(path).absolute()
    if any(p.is_symlink() or getattr(p, 'is_junction', lambda: False)()
           for p in (path, *path.parents)):
        raise ValueError('La carpeta de transporte contiene enlaces no permitidos.')
    return path
bootstrap_path(REMOTE_PACKAGE_DIR)
transport_path = bootstrap_path(REMOTE_PACKAGE_DIR / 'transport.json')
if not transport_path.is_file() or not 0 < transport_path.stat().st_size <= 1024 * 1024:
    raise ValueError('No se encontró un manifiesto de transporte válido.')
transport_raw = transport_path.read_bytes()
transport_manifest = json.loads(transport_raw)
if transport_manifest.get('version') != 'colab_development_transport_v1.0':
    raise ValueError('La versión del transporte no coincide.')
module_spec = transport_manifest.get('transport_module', {})
if (set(module_spec) != {'file', 'sha256', 'bytes'}
        or module_spec.get('file') != 'colab_package_transport.py'
        or type(module_spec.get('bytes')) is not int or not 0 < module_spec['bytes'] <= 1024 * 1024
        or re.fullmatch('[0-9a-f]{64}', str(module_spec.get('sha256'))) is None):
    raise ValueError('La referencia del módulo de transporte no es válida.')
module_path = bootstrap_path(REMOTE_PACKAGE_DIR / module_spec['file'])
if not module_path.is_file() or module_path.stat().st_size != module_spec['bytes']:
    raise ValueError('El módulo de transporte está incompleto.')
module_raw = module_path.read_bytes()
if hashlib.sha256(module_raw).hexdigest() != module_spec['sha256']:
    raise ValueError('La integridad del módulo de transporte no coincide.')
# Se ejecutan únicamente los bytes del módulo cuyo hash ya se comprobó.
bootstrap_spec = importlib.util.spec_from_file_location('knee_transport_bootstrap', module_path)
transport_module = importlib.util.module_from_spec(bootstrap_spec)
exec(compile(module_raw, str(module_path), 'exec'), transport_module.__dict__)
LOCAL_PACKAGE_DIRECTORY = tempfile.TemporaryDirectory(prefix='knee_development_transport_')
LOCAL_PACKAGE_DIR = Path(LOCAL_PACKAGE_DIRECTORY.name)
transport_result = transport_module.stage_development_package(REMOTE_PACKAGE_DIR, LOCAL_PACKAGE_DIR)
if (REMOTE_PACKAGE_DIR / 'transport.json').read_bytes() != transport_raw:
    raise ValueError('El manifiesto de transporte cambió durante la comprobación.')
print('Los fragmentos se reconstruyeron y verificaron en el almacenamiento temporal de CPU.')


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, stat, sys, tempfile, zipfile

PACKAGE_DIR = LOCAL_PACKAGE_DIR
if not PACKAGE_DIR.is_dir() or any(p.is_symlink() for p in (PACKAGE_DIR, *PACKAGE_DIR.parents)):
    raise ValueError('La carpeta indicada no existe o contiene enlaces.')


In [ ]:
def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def require(condition, message):
    if not condition:
        raise ValueError(message)

handoff_path = PACKAGE_DIR / 'handoff.json'
require(handoff_path.is_file() and not handoff_path.is_symlink() and handoff_path.stat().st_size <= 1024 * 1024,
        'No se encontró un registro de entrega válido.')
handoff = json.loads(handoff_path.read_text(encoding='utf-8'))
require(handoff.get('version') == 'participant_split_colab_preparation_v1.0'
        and handoff.get('training_executed') is False
        and handoff.get('reserved_test_predictive_evaluation_opened') is False,
        'El registro no corresponde a la preparación autorizada.')
require(handoff['development']['file'] == 'development_package.zip'
        and handoff['reserved_test']['file'] == 'reserved_test_package.zip'
        and handoff['code']['file'] == 'colab_preparation_code.zip',
        'Los nombres de los paquetes no coinciden con la entrega.')

ALLOWED_CODE_FILES = {
    'src/knee/__init__.py', 'src/knee/colab_data.py', 'src/knee/colab_data_package.py',
    'src/knee/participant_partitions.py', 'scripts/verify_colab_data.py',
    'configs/participant_split_v1.json', 'requirements-colab-data.txt'}
require(set(handoff['code']['files_sha256']) == ALLOWED_CODE_FILES,
        'La lista de archivos de código no coincide.')
code_path = PACKAGE_DIR / handoff['code']['file']
require(code_path.is_file() and not code_path.is_symlink()
        and file_sha(code_path) == handoff['code']['sha256'], 'El paquete de código cambió.')
CODE_DIRECTORY = tempfile.TemporaryDirectory(prefix='knee_colab_data_')
code_root = Path(CODE_DIRECTORY.name)
with zipfile.ZipFile(code_path) as archive:
    entries = archive.infolist()
    require(len(entries) == len(ALLOWED_CODE_FILES)
            and {e.filename for e in entries} == ALLOWED_CODE_FILES, 'El contenido del código no coincide.')
    for entry in entries:
        require(not entry.is_dir() and 0 < entry.file_size <= 1024 * 1024
                and not stat.S_ISLNK(entry.external_attr >> 16)
                and stat.S_IFMT(entry.external_attr >> 16) in {0, stat.S_IFREG}
                and not entry.flag_bits & 1, 'El código contiene un archivo no permitido.')
        raw = archive.read(entry)
        require(hashlib.sha256(raw).hexdigest() == handoff['code']['files_sha256'][entry.filename],
                'La integridad de un archivo de código no coincide.')
        destination = code_root.joinpath(*entry.filename.split('/'))
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open('xb') as stream:
            stream.write(raw)
require(file_sha(code_path) == handoff['code']['sha256'], 'El código cambió durante su lectura.')
# Importación desde la copia verificada; el entorno no instala modelos ni PyTorch.
require('knee' not in sys.modules, 'Reinicia el entorno para evitar un módulo Knee cargado previamente.')
sys.path.insert(0, str(code_root / 'src'))
from knee.colab_data import verify_development_package
print('Código de verificación comprobado. Solo se abrirá el paquete de desarrollo.')

In [ ]:
development = handoff['development']
record_path = PACKAGE_DIR / 'colab_development_verification.json'
summary = verify_development_package(
    PACKAGE_DIR / development['file'], development['sha256'], record_path=record_path)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Comprobación guardada. No se entrenó ni se abrió el paquete reservado de prueba.')

In [ ]:
FINAL_PREFLIGHT = {
    'version': 'colab_development_transport_preflight_v1.1',
    'status': 'passed',
    'colab_runtime_executed': bool(os.environ.get('COLAB_RELEASE_TAG')) and '_LOCAL_REHEARSAL' not in globals(),
    'local_rehearsal': globals().get('_LOCAL_REHEARSAL', False) is True,
    'cpu_only_guard_passed': True,
    'development_package_sha256': transport_result['sha256'],
    'development_package_bytes': transport_result['bytes'],
    'transport_parts_verified': transport_result['parts'],
    'verified_development_participants': summary['participants'],
    'verified_development_tensors': summary['verified_tensors'],
    'all_tensor_file_and_pixel_hashes_verified': summary['all_tensor_file_and_pixel_hashes_verified'],
    'reserved_test_archive_present_in_local_stage': (LOCAL_PACKAGE_DIR / 'reserved_test_package.zip').exists(),
    'reserved_test_predictive_evaluation_opened': False,
    'training_executed': False,
    'imputation_performed': False,
    'model_specific_normalization_applied': False
}
report_root = LOCAL_PACKAGE_DIR if globals().get('_LOCAL_REHEARSAL', False) else REMOTE_PACKAGE_DIR
FINAL_REPORT_PATH = report_root / ('colab_development_preflight_' + uuid.uuid4().hex + '.json')
with FINAL_REPORT_PATH.open('x', encoding='utf-8') as stream:
    json.dump(FINAL_PREFLIGHT, stream, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)
    stream.write('\n')
print(json.dumps(FINAL_PREFLIGHT, ensure_ascii=False, indent=2))
print('Comprobación terminada. No se entrenó ni se abrió el paquete reservado de prueba.')


Comparte únicamente el JSON final. Cuando la libreta se ejecuta realmente en Colab, se guarda una comprobación nueva en la carpeta de transporte de Drive. Un ensayo local se identifica expresamente y guarda su registro solo en almacenamiento temporal local.